# INSTRUCTOR GUIDE: Week 7 Session 1 ML workflow + Linear and Logistic Regression
Not for students. Solutions are filled in below each Your turn. Keep `greenline_deliveries.csv` in the same folder as this notebook.

## Run of show (2 hours)
| Time | Block | What happens |
|---|---|---|
| 0:00 to 0:10 | Check-in and mission | SQL challenge in? Capstone thought started? Today is a new tool, but the habit is the same: question first, honest check, then a model. |
| 0:10 to 0:30 | Concept: the ML workflow and why a baseline comes first | Walk the nine steps on the board. Supervised learning, features vs target, leakage, train/test, in plain words. No algorithm talk yet. |
| 0:30 to 0:55 | Live coding: load, inspect, prep, split, baseline, linear regression | Interpret MAE against the baseline (about 16.8 vs 3.9 minutes). Read a few coefficients aloud. Your turn 1 to 3. |
| 0:55 to 1:05 | Break | |
| 1:05 to 1:40 | Logistic regression for late deliveries | Dummy accuracy of 0.75 is the trap. Then fit logistic and compare. Your turn 4. Precision and recall are named today; Session 2 goes deeper. |
| 1:40 to 1:55 | Challenge | A short coefficient-reading and a first logistic run if anyone is behind. |
| 1:55 to 2:00 | Recap and Introductory ML Project brief | Workflow first. They may use this courier data, `used_phones.csv`, or another small table. Capstone continues in parallel. |

**If time runs short:** keep the baseline vs linear contrast and the 75 percent dummy trap. Move the scatter plot or Your turn 4 to homework.

## Talking points
- Workflow first, algorithms second: Problem, Features and Target, Prep, Split, Baseline, Model, Predict, Evaluate, Interpret.
- A model that is not compared with a baseline is incomplete. The baseline for delivery time is "guess the training-set average for every new drop".
- Train/test split is how we stop the model from being marked on questions it has already seen. `random_state=42` makes the split repeatable.
- Leakage: information that would not be known at prediction time, or that already contains the answer. `delivery_minutes` must not be a feature for `arrived_late`.
- Linear regression: a weighted sum of the features. A coefficient is "how much the prediction moves if this feature goes up by one, holding the others still".
- Logistic regression: same idea, but the target is 0 or 1 (on time or late). The output we care about today is the predicted class and whether it beats a dummy that always says "on time".
- About 25 percent of drops are late, so a dummy that always predicts on time is right 75 percent of the time and never catches a late drop. Accuracy alone is not enough. That sentence is the bridge into Session 2.
- Keep interpretation in the foreground. "Outskirts adds about 8 minutes compared with Mainland" is the kind of sentence we want.

## Common mistakes to watch for
- Fitting a model before a baseline, or skipping the train/test split.
- Filling missing values after the split using the test median (mild leakage). In this intro class we fill with the overall median before the split because only 8 rows are missing; mention the stricter habit (fit the median on the training set only) as a later refinement, not a rabbit hole.
- Using `delivery_minutes` as a feature for `arrived_late`.
- Overwriting the raw table instead of `work = raw.copy()`.
- Forgetting to drop one area column, or dropping the wrong one and then misreading coefficients.
- Reading a coefficient as "this causes the outcome" rather than "this is the change in the prediction, holding the other features still".
- Celebrating 75 percent accuracy on the dummy.
- Running cells out of order. Restart and Run All if it breaks.

## Expected numbers (GreenLine, random_state=42, test_size=0.25)
- 400 rows, 8 missing `package_kg`. Median used to fill: 8.75.
- Late share: 0.248 (99 late, 301 on time). Areas: Mainland 191, Island 126, Outskirts 83.
- Mean delivery time about 60.0 minutes (median 60.8).
- Split: 300 train, 100 test. Training mean of delivery time about 59.53 minutes.
- Dummy regressor: MAE 16.80, RMSE 19.13, R-squared -0.009 (a mean baseline often has R-squared near 0, and a little below 0 on the test set is normal).
- Linear regression: MAE 3.87, RMSE 4.76, R-squared 0.938. Intercept about 6.90.
- Coefficients, rounded: distance_km 2.786; package_kg 0.927; is_weekend 6.67; rain_mm 0.779; depart_hour 0.661; rider_experience_months -0.119; area_Island 4.032; area_Outskirts 8.449.
- Classification test set: 25 late, 75 on time.
- Dummy classifier (most frequent): accuracy 0.75, precision 0, recall 0. Confusion matrix [[75, 0], [25, 0]].
- Logistic regression: accuracy 0.88, precision 0.741, recall 0.80. Confusion matrix [[68, 7], [5, 20]] — 20 late drops caught, 5 missed, 7 false alarms.


# Week 7 Session 1: The ML workflow, then Linear and Logistic Regression
**Forge Academy | Data Science Foundations**

*Covers Week 7 Session 1: the machine learning workflow, a baseline, Linear Regression and Logistic Regression, using scikit-learn at an introductory level.*

## Mission

GreenLine Courier promises customers a delivery time. Promises that are too tight create late drops; promises that are too loose lose business. Our mission today is to make an honest estimate of how long a drop will take, and then to ask a second question: will this drop be late?

We will not start by fitting a fancy model. We will follow a fixed workflow, and we will only trust a model that beats a simple baseline.

Keep `greenline_deliveries.csv` in the same folder as this notebook, open that folder in VS Code, and run the cells from top to bottom.


## 0. The workflow, in plain words

Supervised machine learning is a way to learn a rule from examples where we already know the answer, and then use that rule on new examples.

The workflow we will use every time is:

1. **Problem** — what decision will this help, in one sentence?
2. **Features and target** — which columns do we already know (features), and which column are we trying to estimate (the target)?
3. **Prep** — inspect, clean, and shape the table so a model can use it.
4. **Split** — hold some rows back as a test set, so we can mark the model on rows it has not trained on.
5. **Baseline** — a simple guess (today: the average time, or "always on time") that any real model must beat.
6. **Model** — fit the model on the training rows only.
7. **Predict** — use the fitted model on the test rows.
8. **Evaluate** — score the predictions with a metric that matches the problem.
9. **Interpret** — say what the numbers mean, and what they do not mean, in plain words.

Two warnings we will keep returning to:

- **No baseline means the work is incomplete.** A low error looks impressive until you see that guessing the average was almost as good.
- **Leakage** means the model was given information it would not honestly have at prediction time, or information that already contains the answer. We will be careful with that today.


## 1. Load and inspect the deliveries


In [ ]:
import pandas as pd


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix


In [ ]:
raw = pd.read_csv("greenline_deliveries.csv")


In [ ]:
raw.head()


In [ ]:
print(raw.shape)


In [ ]:
raw.info()


In [ ]:
raw.describe()


In [ ]:
raw["area"].value_counts()


In [ ]:
raw["arrived_late"].value_counts()


In [ ]:
print(raw["arrived_late"].mean())


In [ ]:
raw.isna().sum()


Each row is one delivery.

- The **regression target** is `delivery_minutes`: how long the drop took.
- The **classification target** is `arrived_late`: 1 if it was late, 0 if it was on time.
- The features we are allowed to use are the things GreenLine would know when the package goes out: distance, weight, weekend or not, rain, departure hour, the rider's experience, and the area (Mainland, Island, Outskirts).

You should see 400 rows and 8 missing values in `package_kg`. About a quarter of the drops are late (the mean of a 0/1 column is the late rate). We will come back to that rate when we talk about accuracy.


**INSTRUCTOR NOTE.** Shape (400, 10). `package_kg` has 8 missing; every other column is complete. Area: Mainland 191, Island 126, Outskirts 83. `arrived_late`: 301 zeros, 99 ones, mean 0.2475 (report as 24.8 percent). Mean `delivery_minutes` 59.98, median 60.8. Ask someone to say the problem in one sentence before you prep.


### Your turn 1

1. Print the median of `package_kg` (pandas will ignore the missing values).
2. Print the mean of `delivery_minutes`.
3. In a sentence: why would it be unfair to use `delivery_minutes` as a feature when we try to predict `arrived_late`?


In [ ]:
# SOLUTION
print(raw["package_kg"].median())


In [ ]:
# SOLUTION
print(raw["delivery_minutes"].mean())


**SOLUTION — part 3.** Late deliveries are the long ones. If the model is given the actual duration, it can almost read off whether the drop was late. At the moment GreenLine has to make a promise, that duration is not known yet. Using it would be leakage.

**INSTRUCTOR NOTE. Your turn 1.** Median package weight **8.75** kg. Mean delivery time about **59.98** minutes. Collect two or three wordings of the leakage sentence; they do not have to be identical.


## 2. Prepare a modelling table


We will use the same preparation in both Week 7 sessions, so this notebook stands on its own.

The habit is: work on a copy, fill the small number of missing package weights with the median, and turn `area` into 0/1 columns. We drop `area_Mainland` and treat Mainland as the reference area. Coefficients for Island and Outskirts will then mean "compared with Mainland".

Do not put `delivery_minutes` into the feature list when the target is `arrived_late`. Late deliveries are the long ones, so that would leak the answer.


In [ ]:
work = raw.copy()


In [ ]:
package_median = work["package_kg"].median()


In [ ]:
package_median


In [ ]:
work["package_kg"] = work["package_kg"].fillna(package_median)


In [ ]:
print(work["package_kg"].isna().sum())


In [ ]:
area_flags = pd.get_dummies(work["area"], prefix="area")


In [ ]:
area_flags.head()


In [ ]:
area_flags = area_flags.drop(columns=["area_Mainland"])


In [ ]:
area_flags.head()


In [ ]:
work = pd.concat([work, area_flags], axis=1)


In [ ]:
work.head()


`pd.get_dummies` builds a 0/1 column for each area. We drop Mainland so the model has a reference. Island and Outskirts will be read as "compared with Mainland".

Now we name the feature list and the two targets. Notice that `delivery_minutes` and `arrived_late` are not in the feature list.


In [ ]:
feature_names = [
    "distance_km",
    "package_kg",
    "is_weekend",
    "rain_mm",
    "depart_hour",
    "rider_experience_months",
    "area_Island",
    "area_Outskirts",
]


In [ ]:
X = work[feature_names]


In [ ]:
y_time = work["delivery_minutes"]


In [ ]:
y_late = work["arrived_late"]


In [ ]:
X.head()


## 3. Split, then a baseline, then Linear Regression


`train_test_split` shuffles the rows and holds some back. We use 25 percent for the test set and `random_state=42` so everyone gets the same split. The four results are: training features, test features, training target, test target.


In [ ]:
X_train, X_test, y_time_train, y_time_test = train_test_split(X, y_time, test_size=0.25, random_state=42)


In [ ]:
print(len(X_train))


In [ ]:
print(len(X_test))


In [ ]:
print(y_time_train.mean())


In [ ]:
print(y_time_test.mean())


The baseline for a "how long will this take?" question is: guess the average training time for every test drop. `DummyRegressor(strategy="mean")` does exactly that.


In [ ]:
time_baseline = DummyRegressor(strategy="mean")


In [ ]:
time_baseline.fit(X_train, y_time_train)


In [ ]:
y_time_baseline = time_baseline.predict(X_test)


In [ ]:
print(mean_absolute_error(y_time_test, y_time_baseline))


In [ ]:
print(mean_squared_error(y_time_test, y_time_baseline) ** 0.5)


In [ ]:
print(r2_score(y_time_test, y_time_baseline))


**MAE** (mean absolute error) is the average size of the mistake, in minutes, ignoring sign. **RMSE** (root mean squared error) is also in minutes, but it penalises large mistakes more. **R-squared** says how much of the variation in the target the model explains; a simple mean baseline is usually near 0.

Now fit Linear Regression on the same training rows, predict the test rows, and compare.


In [ ]:
time_model = LinearRegression()


In [ ]:
time_model.fit(X_train, y_time_train)


In [ ]:
y_time_pred = time_model.predict(X_test)


In [ ]:
print(mean_absolute_error(y_time_test, y_time_pred))


In [ ]:
print(mean_squared_error(y_time_test, y_time_pred) ** 0.5)


In [ ]:
print(r2_score(y_time_test, y_time_pred))


In [ ]:
print(time_model.intercept_)


In [ ]:
coef_table = pd.DataFrame({"feature": feature_names, "coefficient": time_model.coef_})


In [ ]:
coef_table


Read the coefficients slowly.

- `distance_km` is about 2.8: each extra kilometre adds about 3 minutes, holding the other features still.
- `is_weekend` is about 6.7: a weekend drop is predicted to take roughly 7 minutes longer than a weekday drop with the same distance, weight, and so on.
- `area_Outskirts` is about 8.4: compared with Mainland, an outskirts drop is predicted to take about 8 minutes longer.
- `rider_experience_months` is slightly negative: more experienced riders are predicted to be a little quicker.

These are associations in the model, not automatic proof of cause. Still, they should sound plausible.

A quick chart: each test drop is one point, actual time on the x-axis, predicted time on the y-axis. A strong model will hug the diagonal.


In [ ]:
plt.figure()
plt.scatter(y_time_test, y_time_pred, alpha=0.7)
plt.xlabel("Actual delivery minutes")
plt.ylabel("Predicted delivery minutes")
plt.title("Linear regression: predicted versus actual time")
plt.show()


**INSTRUCTOR NOTE.** Train 300, test 100. Training mean about 59.53, test mean will be close but not identical. Baseline MAE **16.80**, RMSE **19.13**, R-squared **-0.009**. Linear MAE **3.87**, RMSE **4.76**, R-squared **0.938**, intercept about **6.90**. The model is clearly better than guessing the average, and the relationship in this teaching set is deliberately strong. Say that out loud so no one thinks every real problem will give R-squared of 0.94. Largest coefficient is `area_Outskirts` at about 8.45; among the 0/1 flags, weekend is about 6.67. Experience is a small negative.


### Your turn 2

1. Print the baseline MAE and the linear regression MAE again, next to each other in whatever way you like (two print lines is fine).
2. Which feature has the largest coefficient in `coef_table`? You can sort the table or just read it.
3. Write two sentences: did the linear model beat the baseline, and what does the outskirts coefficient mean in plain words?


In [ ]:
# SOLUTION
print(mean_absolute_error(y_time_test, y_time_baseline))
print(mean_absolute_error(y_time_test, y_time_pred))


In [ ]:
# SOLUTION
coef_table.sort_values("coefficient", ascending=False)


**SOLUTION — part 3.** Yes: the average mistake falls from about 16.8 minutes to about 3.9 minutes. The outskirts coefficient (about 8.4) means that, holding the other features still, a drop in the Outskirts is predicted to take roughly 8 minutes longer than a similar drop on the Mainland.

**INSTRUCTOR NOTE. Your turn 2.** If they sort ascending, outskirts may sit at the bottom of a list that also has negative experience; ask for the largest *positive* coefficient, or the largest in size if someone mentions experience. Either is a good conversation.


## 4. A second question: will this drop be late?

Same features, new target: `arrived_late`. This is a **classification** problem (two classes: on time or late).

We split again, with the same `random_state`, and we add `stratify=y_late` so the training set and the test set have about the same late rate.


In [ ]:
X_train_c, X_test_c, y_late_train, y_late_test = train_test_split(X, y_late, test_size=0.25, random_state=42, stratify=y_late)


In [ ]:
print(len(X_train_c))


In [ ]:
print(len(X_test_c))


In [ ]:
print(y_late_train.mean())


In [ ]:
print(y_late_test.mean())


In [ ]:
print(y_late_test.sum())


The baseline this time is `DummyClassifier(strategy="most_frequent")`: always predict the more common class. Here that means always predict "on time".


In [ ]:
late_baseline = DummyClassifier(strategy="most_frequent")


In [ ]:
late_baseline.fit(X_train_c, y_late_train)


In [ ]:
y_late_baseline = late_baseline.predict(X_test_c)


In [ ]:
print(accuracy_score(y_late_test, y_late_baseline))


In [ ]:
print(precision_score(y_late_test, y_late_baseline, zero_division=0))


In [ ]:
print(recall_score(y_late_test, y_late_baseline, zero_division=0))


In [ ]:
print(confusion_matrix(y_late_test, y_late_baseline))


**Accuracy** is the share of rows the model got right. The dummy is right on about 75 percent of test drops, because 75 of the 100 test drops are on time. That sounds good until you notice it never catches a late drop.

**Precision** asks: of the drops we *called* late, how many really were late? **Recall** asks: of the drops that *really were* late, how many did we catch? The dummy predicts no late drops at all, so both precision and recall are 0.

The **confusion matrix** is a 2-by-2 count, in this order:

```
[[true on-time,  false late],
 [false on-time, true late]]
```

For the dummy it is `[[75, 0], [25, 0]]`: every late drop is a miss.

This is why "accuracy only" is dangerous on a problem where one class is more common than the other. Session 2 will spend more time on these metrics. Today we fit Logistic Regression and ask whether it beats this dummy in a more honest way.


In [ ]:
late_model = LogisticRegression(max_iter=1000, random_state=42)


In [ ]:
late_model.fit(X_train_c, y_late_train)


In [ ]:
y_late_pred = late_model.predict(X_test_c)


In [ ]:
print(accuracy_score(y_late_test, y_late_pred))


In [ ]:
print(precision_score(y_late_test, y_late_pred, zero_division=0))


In [ ]:
print(recall_score(y_late_test, y_late_pred, zero_division=0))


In [ ]:
print(confusion_matrix(y_late_test, y_late_pred))


Logistic regression should land near 88 percent accuracy, with precision about 0.74 and recall about 0.80. In the confusion matrix you should see most late drops caught (around 20 out of 25) and a small number of false alarms. That is already more useful than a rule that never raises a hand.

We will compare this model with a Decision Tree in Session 2, and we will practise saying what the matrix means in a sentence.


**INSTRUCTOR NOTE.** Test late rate 0.25 (25 rows). Dummy: accuracy 0.75, precision 0, recall 0, matrix [[75, 0], [25, 0]]. Logistic: accuracy 0.88, precision 0.741, recall 0.80, matrix [[68, 7], [5, 20]]. Walk the matrix in words: 68 correctly on time, 20 correctly late, 7 false late alarms, 5 late drops missed. Ask "which error would bother GreenLine more?" There is no single right business answer, but the question is the point.


### Your turn 3

1. Print the dummy accuracy and the logistic accuracy.
2. How many late test drops did the logistic model catch, and how many did it miss? Read them from the confusion matrix (bottom row).
3. Write two sentences: why is the dummy's 75 percent accuracy not a good enough report, and what does a recall of about 0.80 mean here?


In [ ]:
# SOLUTION
print(accuracy_score(y_late_test, y_late_baseline))
print(accuracy_score(y_late_test, y_late_pred))


In [ ]:
# SOLUTION
print(confusion_matrix(y_late_test, y_late_pred))


**SOLUTION — parts 2 and 3.** The bottom row of `[[68, 7], [5, 20]]` is 5 misses and 20 catches, out of 25 late test drops. The dummy is right 75 percent of the time only because most drops are on time; it never identifies a late one. A recall of about 0.80 means the logistic model caught four fifths of the drops that were actually late.

**INSTRUCTOR NOTE. Your turn 3.** If someone swaps "catch" and "miss", go back to the matrix layout and redraw it on a whiteboard.


## 5. Challenge (10 to 15 minutes)

1. Look at `coef_table` from the linear model. Write one sentence about `is_weekend` and one sentence about `rider_experience_months`.
2. Using the same training and test classification sets we already created, print the logistic model's precision and recall again, then write one sentence about which of those two you would watch more carefully if a late delivery was very costly for GreenLine.
3. Optional stretch: print `late_model.coef_` next to `feature_names` (the same pattern as `coef_table`) and name one feature that the logistic model associates with a higher chance of being late.


In [ ]:
# SOLUTION — parts 1 and 2 use numbers already computed
print(coef_table)
print(precision_score(y_late_test, y_late_pred, zero_division=0))
print(recall_score(y_late_test, y_late_pred, zero_division=0))


In [ ]:
# SOLUTION — optional stretch
late_coef_table = pd.DataFrame({"feature": feature_names, "coefficient": late_model.coef_[0]})
late_coef_table


**SOLUTION — sentences.** Weekend drops are predicted to take about 7 minutes longer than similar weekday drops. More experienced riders are predicted to be slightly quicker (the coefficient is a small negative number). If a late delivery is very costly, recall is the number to watch, because it is the share of actual late drops that we catch. In the logistic coefficients, larger positive values (distance, outskirts, weekend) are associated with a higher chance of being late.

**INSTRUCTOR NOTE. Challenge.** Logistic coefficients are not in minutes, so do not let students read them as "adds 0.6 minutes". They are in a log-odds scale we are not teaching formally. "This feature is associated with a higher predicted chance of being late" is enough.


## Recap

- Follow the workflow in order: Problem, Features and Target, Prep, Split, Baseline, Model, Predict, Evaluate, Interpret.
- A baseline is not optional. Today that was the mean delivery time, and "always on time".
- Linear regression predicts a number. MAE, RMSE and R-squared tell us whether it beats the baseline, and coefficients are a first interpretation.
- Logistic regression predicts a class. Accuracy is a poor headline when one class is more common; look at precision, recall and the confusion matrix as well.
- Do not use information that leaks the answer. `delivery_minutes` is not a legal feature for `arrived_late`.
- `random_state=42` makes the split and the model run repeatable.

Next session we will fit a Decision Tree, compare it with today's models, and practise reading the metrics until the sentences feel natural. That comparison is also the core of the Introductory ML Project.


## Assignment (start the Introductory ML Project)

**Due:** the project itself is due before our Week 8 Session 1 class; this week, start it and bring questions to Session 2. **Submit** the finished notebook through the usual submission form, when it is complete.

The full brief is `Week7_Introductory_ML_Project.txt` in this folder.

In short: complete a small supervised ML project with scikit-learn and honest evaluation. Follow the full workflow. You must have a train/test split and a **baseline before the model**. Compare a linear or logistic model with a Decision Tree (you can finish the tree after Session 2). Interpret the metrics in plain language, and add a short limitations note. Please also add a one-paragraph capstone progress update (a parallel track, not part of this project's score).

You may use `greenline_deliveries.csv`, the optional `used_phones.csv` in this folder (for example, predict `sold_price` or `days_to_sell`), or another small tabular dataset that you can explain.

**Rubric (the project is 10 percent of the course):** Technical correctness, Problem solving, Analytical reasoning, Interpretation, Reproducibility, Documentation. Missing baseline or obvious leakage will be penalised.

AI tools are allowed, but you must be able to explain every line and every metric on our call.
